# UK Road Safety Command Platform
### From raw police records to an interactive danger-zone dashboard

**Why this project?** Road collisions are not random. Where a road bends, how a junction is laid out, what the speed limit is, whether the street is lit, and what the weather is doing all shape where people get hurt. This notebook turns the official UK police collision records (STATS19, 2025) into an interactive command-center dashboard that shows *where* the danger concentrates and *why*.

**Dataset.** Road Safety Data published by the UK Department for Transport: 101,525 reported collisions, 183,948 vehicles and 127,883 casualties across Great Britain, with the road type, junction, speed limit, lighting, weather, surface and GPS location of every collision.

**What you will build**

| Step | What happens |
|---|---|
| 1. Load | You place the three CSV files (collision, vehicle, casualty) in the Files panel |
| 2. Explore | Shapes, data types, missing values, `-1` codes, value distributions, date and coordinate ranges, table linkage |
| 3. Clean | Unknown codes become `Unknown`, dates and times are validated, the 2025 junction coding change is handled |
| 4. Merge | Vehicles and casualties are folded into one row per collision |
| 5. Analyse | Severity by junction type and other quick checks |
| 6. Build | A single self-contained HTML dashboard is generated |
| 7. Download | `uk_road_safety_dashboard.html` is saved to your computer |

**What the dashboard offers**
- A zoomable map of Great Britain with every collision, coloured by severity (fatal, serious, slight)
- Pulsing red markers on the most dangerous zones, re-ranked as you zoom and filter
- Click any area for a full report: collisions, casualties, and what stands out against the national average
- A street-level replay of how collisions happen, built from the recorded road layout, speed limit, light, weather and road user
- Four tabs (Overview, Road design, Time, Conditions) with charts that cross-filter the map and each other

**How to run:** upload the three CSV files to the Colab **Files** panel (left sidebar), then choose **Runtime > Run all**. No other setup is needed.

**Good to know.** The street-level replay is an illustrative reconstruction: the police record gives the road layout and conditions but not the exact vehicle paths. The data covers Great Britain only and reports police-recorded collisions, so minor incidents are under-represented.

In [ ]:
!pip -q install pandas numpy
import os
from google.colab import files   # used at the end to download the dashboard

# The three data files, uploaded to the Colab "Files" panel (left sidebar)
COLLISION_FILE = 'dft-road-casualty-statistics-collision-2025.csv'
VEHICLE_FILE   = 'dft-road-casualty-statistics-vehicle-2025.csv'
CASUALTY_FILE  = 'dft-road-casualty-statistics-casualty-2025.csv'

for f in [COLLISION_FILE, VEHICLE_FILE, CASUALTY_FILE]:
    if os.path.exists(f):
        print('OK     ', f, f'({os.path.getsize(f)/1e6:.1f} MB)')
    else:
        print('MISSING', f, '-> upload it to the Files panel on the left, then run this cell again')

The next two cells save the dashboard template and the Great Britain outline next to the data.

In [ ]:
%%writefile template.html
<!DOCTYPE html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<title>UK Road Safety Command Platform</title>
<style>
:root{color-scheme:dark;--bg:#040a07;--cd:#09130e;--ln:#164a35;--g:#2dff8f;--tx:#d3f5e5;--mu:#6b9983;--r:#ff4d5a;--a:#ffb020;box-sizing:border-box;padding-top:env(safe-area-inset-top,0px);padding-bottom:env(safe-area-inset-bottom,0px)}
*{box-sizing:border-box}html,body{height:100%;margin:0}
body{background:var(--bg);color:var(--tx);font:12px/1.3 "Segoe UI",system-ui,Arial,sans-serif}
#app{height:100%;display:grid;grid-template-columns:1fr 310px;grid-template-rows:54px 1fr 205px;gap:8px;padding:8px}
header{grid-column:1/3;display:flex;align-items:center;gap:18px;border:1px solid var(--ln);background:linear-gradient(90deg,#0b2a1c,var(--cd));padding:6px 14px;border-radius:6px;flex-wrap:wrap}
h1{font-size:16px;margin:0;color:var(--g);white-space:nowrap}
nav{display:flex;gap:4px;flex-wrap:wrap}
button{font:inherit;cursor:pointer}
nav button{background:none;border:1px solid transparent;color:var(--mu);padding:6px 12px;border-radius:4px}
nav button.on{color:var(--g);border-color:var(--g);background:#0d2a1c}
#chips{display:flex;gap:6px;flex-wrap:wrap;margin-left:auto}
#chips button,.btn{background:#0d2a1c;border:1px solid var(--ln);color:var(--tx);border-radius:12px;padding:3px 10px}
.btn:hover,#chips button:hover{border-color:var(--g)}
#mapw{grid-row:2;position:relative;border:1px solid var(--ln);border-radius:6px;overflow:hidden;background:radial-gradient(circle at 50% 45%,#0d2b1d,#040a07 72%)}
#mapw:before{content:"";position:absolute;inset:0;background-image:linear-gradient(#123d2b44 1px,transparent 1px),linear-gradient(90deg,#123d2b44 1px,transparent 1px);background-size:40px 40px}
#cv,#ov,#sc{position:absolute;inset:0;width:100%;height:100%}
#ov{cursor:grab}
#lg{position:absolute;left:12px;bottom:10px;display:flex;gap:12px;color:var(--mu);background:#040a07cc;padding:4px 8px;border-radius:4px}
#lg i{display:inline-block;width:8px;height:8px;border-radius:50%;margin-right:4px}
#src{position:absolute;right:12px;bottom:10px;color:var(--mu);text-align:right;max-width:50%;pointer-events:none}
#zb{position:absolute;left:12px;top:12px;display:flex;flex-direction:column;gap:4px}
#zb button{width:28px;height:28px;background:#06120dcc;border:1px solid var(--ln);color:var(--g);border-radius:4px;font-size:16px}
#hl{position:absolute;left:52px;top:12px;width:210px;background:#06120dd9;border:1px solid var(--ln);border-radius:6px;padding:8px}
#hl h3{margin:0 0 6px;color:var(--r);font-size:12px}
#hl div{display:flex;justify-content:space-between;gap:8px;padding:3px 0;cursor:pointer;border-top:1px solid #10281d}
#hl div:hover{color:var(--g)}#hl em{font-style:normal;color:var(--r)}
#card{position:absolute;right:10px;top:10px;bottom:10px;width:300px;background:#06120df2;border:1px solid var(--g);border-radius:8px;padding:10px 12px;overflow:auto;box-shadow:0 0 24px #2dff8f44;display:none}
#card h2{margin:0;font-size:15px;color:var(--g)}#card small{color:var(--mu)}
#card h3{margin:10px 0 4px;font-size:12px;color:var(--g)}
.x{position:absolute;right:8px;top:6px;background:none;border:0;color:var(--mu);font-size:16px}
.kg{display:grid;grid-template-columns:1fr 1fr;gap:6px;margin-top:8px}
.kg div{background:#0c1d15;border:1px solid var(--ln);border-radius:5px;padding:5px 8px}.kg b{display:block;font-size:17px;color:var(--g)}
.why div{padding:3px 0;border-top:1px solid #10281d}.why b{color:var(--r)}
.go{width:100%;margin-top:10px;padding:8px;border-radius:6px;background:#2dff8f;color:#03170d;border:0;font-weight:600}
#sim{position:absolute;inset:0;display:none;background:#040a07}
#si{position:absolute;left:10px;top:10px;width:250px;background:#06120de6;border:1px solid var(--g);border-radius:8px;padding:10px 12px;max-height:calc(100% - 20px);overflow:auto}
#si h2{margin:0 0 6px;font-size:14px}#si div.l{display:flex;justify-content:space-between;gap:10px;padding:3px 0;border-top:1px solid #10281d}#si span{color:var(--mu)}
#si p{color:var(--mu);margin:8px 0 0;font-size:11px}
#sb{position:absolute;right:10px;top:10px;display:flex;gap:6px}
#side{grid-row:2/4;grid-column:2;display:flex;flex-direction:column;gap:8px;min-height:0}
#bot{grid-row:3;display:grid;grid-template-columns:repeat(4,1fr);gap:8px;min-height:0}
.card{background:var(--cd);border:1px solid var(--ln);border-radius:6px;padding:8px 10px;display:flex;flex-direction:column;min-height:0;overflow:hidden}
#side .card{flex:1}
.card h3{margin:0 0 6px;font-size:12px;color:var(--g);font-weight:600}
.k{display:grid;grid-template-columns:1fr 1fr;gap:8px}
.k .card b{font-size:22px;color:var(--g)}.k small{color:var(--mu)}
.hb{display:flex;flex-direction:column;gap:4px;overflow:auto;flex:1}
.r{display:grid;grid-template-columns:96px 1fr 50px;gap:6px;align-items:center}
.r.cl,.c.cl{cursor:pointer}
.r span{color:var(--mu);white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.r b{height:8px;background:#0f2a1f;border-radius:4px;overflow:hidden}
.r i{display:block;height:100%;background:var(--g);border-radius:4px}
.r em{font-style:normal;text-align:right}
.off{opacity:.3}
.vb{display:flex;align-items:stretch;gap:3px;flex:1;min-height:70px}
.c{flex:1;display:flex;flex-direction:column}
.t{flex:1;position:relative}
.t i{position:absolute;bottom:0;left:0;right:0;background:linear-gradient(var(--g),#0a7a44);border-radius:2px 2px 0 0}
.c u{text-decoration:none;font-size:9px;color:var(--mu);height:12px;text-align:center}
@media(max-width:900px){html,body{height:auto}#app{display:flex;flex-direction:column;height:auto}#mapw{height:480px;flex:none}#bot{grid-template-columns:1fr 1fr}.card{min-height:210px}#card{left:10px;width:auto}#hl{display:none}}
</style></head><body>
<div id="app">
<header><h1>UK Road Safety Command Platform</h1><nav id="nav"></nav><div id="chips"></div></header>
<div id="mapw"><canvas id="cv"></canvas><canvas id="ov"></canvas>
<div id="zb"><button id="zi">+</button><button id="zo">–</button><button id="zr" title="Reset view">⌂</button></div>
<div id="hl"></div>
<div id="lg"><span><i style="background:#ff4d5a"></i>Fatal</span><span><i style="background:#ffb020"></i>Serious</span><span><i style="background:#2dff8f"></i>Slight</span></div>
<div id="src">DfT STATS19 · Great Britain · 2025<br>Scroll to zoom, drag to pan, click anywhere for an area report.</div>
<div id="card"></div>
<div id="sim"><canvas id="sc"></canvas><div id="si"></div><div id="sb"><button class="btn" id="sp">‹ Prev</button><button class="btn" id="sn">Next ›</button><button class="btn" id="sx">Close ✕</button></div></div></div>
<div id="side"><div class="k" id="kpi"></div><div class="card" id="s0"></div><div class="card" id="s1"></div><div class="card" id="s2"></div></div>
<div id="bot"><div class="card" id="b0"></div><div class="card" id="b1"></div><div class="card" id="b2"></div><div class="card" id="b3"></div></div>
</div>
<script>
const D="__DATA__",N=__N__,M=__META__,LAND=__LAND__;
const $=s=>document.querySelector(s);
const b=Uint8Array.from(atob(D),c=>c.charCodeAt(0));
const K=['sev','dow','hr','mo','rt','jn','sp','li','we','su','ur','inv','cas','veh'],C={};
K.forEach((k,i)=>C[k]=b.subarray(i*N,(i+1)*N));
const LA=new Uint16Array(b.buffer,14*N,N),LO=new Uint16Array(b.buffer,16*N,N);
const NM=Object.assign({},M,{dow:['Mon','Tue','Wed','Thu','Fri','Sat','Sun'],mo:'Jan Feb Mar Apr May Jun Jul Aug Sep Oct Nov Dec'.split(' '),hr:[...Array(24).keys()].map(String)});
const L={sev:'Severity',inv:'Road user involved',dow:'Day of week',hr:'Hour of day',mo:'Month',sp:'Speed limit',li:'Light conditions',rt:'Road type',jn:'Junction type',ur:'Area',su:'Road surface',we:'Weather'};
const T=[{n:'Overview',r:['sev','inv','dow'],b:['hr','mo','sp','li']},{n:'Road design',r:['rt','jn','sp'],b:['ur','inv','li','su']},{n:'Time',r:['sev','dow','mo'],b:['hr','li','we','ur']},{n:'Conditions',r:['we','su','li'],b:['rt','jn','sev','inv']}];
const SC=['#ff4d5a','#ffb020','#2dff8f'],KX=.574;
const CITIES=[['London',51.51,-.13],['Birmingham',52.48,-1.9],['Manchester',53.48,-2.24],['Leeds',53.8,-1.55],['Liverpool',53.41,-2.99],['Newcastle',54.98,-1.61],['Sheffield',53.38,-1.47],['Bristol',51.45,-2.59],['Cardiff',51.48,-3.18],['Glasgow',55.86,-4.25],['Edinburgh',55.95,-3.19],['Southampton',50.9,-1.4],['Nottingham',52.95,-1.15],['Plymouth',50.37,-4.14],['Inverness',57.48,-4.22],['Aberdeen',57.15,-2.09],['Norwich',52.63,1.3],['Swansea',51.62,-3.94]];
const RINGS=[];LAND.forEach(o=>o.c.forEach(p=>p.forEach(r=>{const a=r.split(' ').map(s=>s.split(',').map(Number)),f=new Float32Array(a.length*2);a.forEach((q,i)=>{f[2*i]=q[0]*KX;f[2*i+1]=-q[1]});RINGS.push({u:o.id==826,f})})));
const LAT=new Float32Array(N),LON=new Float32Array(N),PX=new Float32Array(N),PY=new Float32Array(N);
for(let i=0;i<N;i++){if(!LA[i]){PX[i]=NaN;continue}LAT[i]=49.5+LA[i]/65535*11.5;LON[i]=-9+LO[i]/65535*11;PX[i]=LON[i]*KX;PY[i]=-LAT[i]}
let F={},tab=0,W=0,H=0,R=1,V={k:1,k0:1,cx:-3.4*KX,cy:-55.35},HS=[],sel=null,rq=0,dr=null;
function pass(i,ex){for(const k in F)if(k!==ex&&!F[k].has(C[k][i]))return false;return true}
function counts(d){const a=new Array(NM[d].length).fill(0);for(let i=0;i<N;i++)if(pass(i,d))a[C[d][i]]++;return a}
function countsS(S,d){const a=new Array(NM[d].length).fill(0);for(const i of S)a[C[d][i]]++;return a}
const BASE={};Object.keys(L).forEach(d=>BASE[d]=counts(d));
const GL={};{const f={dk:i=>C.li[i]>0,jn:i=>C.jn[i]>=1&&C.jn[i]<=5,fast:i=>C.sp[i]>=3&&C.sp[i]<=5,wet:i=>C.su[i]!=0};for(const k in f){let n=0;for(let i=0;i<N;i++)if(f[k](i))n++;GL[k]=n/N}GL.f=f}
function hb(a,d,sel,clk){const mx=Math.max(...a,1),v=d==='hr'||d==='mo';let h='<div class="'+(v?'vb':'hb')+'">';
NM[d].forEach((n,j)=>{if(clk?!BASE[d][j]:!a[j]&&!v)return;const p=a[j]/mx*100,on=!sel||sel.has(j),at=(clk?'data-d="'+d+'" data-j="'+j+'" ':'')+'title="'+n+': '+a[j].toLocaleString()+'"',cl=clk?' cl':'';
h+=v?'<div class="c'+cl+(on?'':' off')+'" '+at+'><div class="t"><i style="height:'+p+'%"></i></div><u>'+(d==='hr'?(j%3?'':n):n[0])+'</u></div>'
:'<div class="r'+cl+(on?'':' off')+'" '+at+'><span>'+n+'</span><b><i style="width:'+p+'%'+(d==='sev'?';background:'+SC[j]:'')+'"></i></b><em>'+a[j].toLocaleString()+'</em></div>'});return h+'</div>'}
function chart(el,d){el.innerHTML='<h3>'+L[d]+'</h3>'+hb(counts(d),d,F[d],true)}
function kpi(){let c=0,k=0,f=0,s=0;for(let i=0;i<N;i++)if(pass(i)){c++;k+=C.cas[i];if(C.sev[i]==0)f++;else if(C.sev[i]==1)s++}
const q=(t,v,x)=>'<div class="card"><small>'+t+'</small><b>'+v.toLocaleString()+'</b><small>'+(x||'&nbsp;')+'</small></div>';
$('#kpi').innerHTML=q('Collisions',c,Math.round(c/N*100)+'% of all')+q('Casualties',k)+q('Fatal',f,c?(f/c*1000).toFixed(1)+' per 1,000':'')+q('Serious',s,c?(s/c*100).toFixed(1)+'% of collisions':'')}
function chips(){let h='';for(const d in F)F[d].forEach(j=>h+='<button data-d="'+d+'" data-j="'+j+'">'+L[d]+': '+NM[d][j]+' ×</button>');$('#chips').innerHTML=h+(h?'<button id="rs">Reset all</button>':'')}
const sx=x=>(x-V.cx)*V.k+W/2,sy=y=>(y-V.cy)*V.k+H/2;
function ring(g,f,dy){g.beginPath();for(let j=0;j<f.length;j+=2){const x=sx(f[j]),y=sy(f[j+1])+dy;j?g.lineTo(x,y):g.moveTo(x,y)}g.closePath()}
function drawMap(){rq=0;if(!W)return;const g=$('#cv').getContext('2d');g.setTransform(R,0,0,R,0,0);g.clearRect(0,0,W,H);
RINGS.forEach(r=>{if(r.u){for(let d=7;d>0;d--){ring(g,r.f,d);g.fillStyle=d>3?'#06301d':'#0a4a2b';g.fill()}ring(g,r.f,0);g.fillStyle='#0b2619';g.fill();g.strokeStyle='#2dff8f99';g.lineWidth=1.2;g.stroke()}else{ring(g,r.f,0);g.fillStyle='#07140e';g.fill();g.strokeStyle='#143d2b';g.lineWidth=1;g.stroke()}});
const z=Math.min(5,1.3*Math.sqrt(V.k/V.k0)),fl=Object.keys(F).length;g.globalCompositeOperation='lighter';
if(fl){g.fillStyle='rgba(45,255,143,.07)';for(let i=0;i<N;i++){const x=sx(PX[i]),y=sy(PY[i]);if(x>-3&&x<W&&y>-3&&y<H)g.fillRect(x-z/2,y-z/2,z,z)}}
[2,1,0].forEach(s=>{g.fillStyle=['rgba(255,77,90,.95)','rgba(255,176,32,.6)','rgba(45,255,143,.3)'][s];const q=z*[2.4,1.7,1.2][s];
for(let i=0;i<N;i++){if(C.sev[i]!=s)continue;const x=sx(PX[i]),y=sy(PY[i]);if(x>-3&&x<W&&y>-3&&y<H&&pass(i))g.fillRect(x-q/2,y-q/2,q,q)}});
g.globalCompositeOperation='source-over';g.font='11px Segoe UI,Arial';g.textBaseline='middle';
CITIES.forEach(c=>{const x=sx(c[2]*KX),y=sy(-c[1]);if(x<0||x>W||y<0||y>H)return;g.fillStyle='#d3f5e5';g.beginPath();g.arc(x,y,2.5,0,6.3);g.fill();g.fillStyle='#a9d8c1';g.fillText(c[0],x+6,y)})}
function req(){if(!rq)rq=requestAnimationFrame(drawMap)}
function near(x,y){let m=1e9,n='';CITIES.forEach(c=>{const d=Math.hypot(c[2]*KX-x,-c[1]-y);if(d<m){m=d;n=c[0]}});return m<.5?'near '+n:'Rural area'}
function hot(){const cs=Math.max(.004,.1/Math.pow(V.k/V.k0,.8)),m=new Map(),x0=V.cx-W/2/V.k,x1=V.cx+W/2/V.k,y0=V.cy-H/2/V.k,y1=V.cy+H/2/V.k;
for(let i=0;i<N;i++){const x=PX[i],y=PY[i];if(!(x>x0&&x<x1&&y>y0&&y<y1)||!pass(i))continue;const key=(Math.floor(LAT[i]/cs)+1000)*100000+Math.floor(LON[i]/cs)+50000;let o=m.get(key);if(!o){o={s:0,n:0,f:0,x:0,y:0};m.set(key,o)}const s=C.sev[i];o.s+=s==0?10:s==1?3:1;o.n++;if(s==0)o.f++;o.x+=x;o.y+=y}
HS=[...m.values()].filter(o=>o.n>=5).sort((a,b)=>b.s-a.s).slice(0,12).map(o=>({x:o.x/o.n,y:o.y/o.n,s:o.s,n:o.n,f:o.f,r:cs*.8}));
$('#hl').innerHTML='<h3>Top danger zones in view</h3>'+HS.slice(0,5).map((h,i)=>'<div data-h="'+i+'"><span>'+(i+1)+'. '+near(h.x,h.y)+'</span><em>'+h.n+' · '+h.f+' fatal</em></div>').join('')}
function update(){kpi();const t=T[tab];t.r.forEach((d,i)=>chart($('#s'+i),d));t.b.forEach((d,i)=>chart($('#b'+i),d));chips();hot();req();if(sel&&!SIM.on)openCard(sel.x,sel.y,sel.r)}
function layout(){const w=$('#mapw').clientWidth,h=$('#mapw').clientHeight;if(!w)return;R=devicePixelRatio||1;W=w;H=h;['cv','ov','sc'].forEach(id=>{const c=$('#'+id);c.width=w*R;c.height=h*R});
V.k0=Math.min(w/(10.6*KX),h/11.1)*.94;if(!V.set){V.k=V.k0;V.set=1}req()}
function zoomAt(f,px,py){const wx=(px-W/2)/V.k+V.cx,wy=(py-H/2)/V.k+V.cy;V.k=Math.max(V.k0,Math.min(V.k0*80,V.k*f));V.cx=wx-(px-W/2)/V.k;V.cy=wy-(py-H/2)/V.k;req();clearTimeout(zoomAt.t);zoomAt.t=setTimeout(hot,160)}
function area(x,y,r){const S=[],r2=r*r;for(let i=0;i<N;i++){const dx=PX[i]-x,dy=PY[i]-y;if(dx*dx+dy*dy<r2&&pass(i))S.push(i)}return S}
const pc=(S,f)=>{let n=0;for(const i of S)if(f(i))n++;return n/S.length};
function openCard(x,y,r){sel={x,y,r};const S=area(x,y,r),el=$('#card');el.style.display='block';
if(!S.length){el.innerHTML='<button class="x" id="cx">✕</button><h2>No collisions here</h2><small>Nothing recorded in this spot with the current filters.</small>';return}
let k=0,f=0,s=0;for(const i of S){k+=C.cas[i];if(C.sev[i]==0)f++;else if(C.sev[i]==1)s++}
const why=[['In the dark',pc(S,GL.f.dk),GL.dk],['At a junction',pc(S,GL.f.jn),GL.jn],['On 50–70 mph roads',pc(S,GL.f.fast),GL.fast],['On a wet or icy road',pc(S,GL.f.wet),GL.wet]].map(w=>'<div>'+w[0]+': '+(w[1]>w[2]*1.25?'<b>':'<span>')+Math.round(w[1]*100)+'%'+(w[1]>w[2]*1.25?'</b>':'</span>')+' <small>(GB '+Math.round(w[2]*100)+'%)</small></div>').join('');
el.innerHTML='<button class="x" id="cx">✕</button><h2>'+near(x,y)+'</h2><small>'+(LAT[S[0]].toFixed(2))+'°N, '+LON[S[0]].toFixed(2)+'° · area report</small>'+
'<div class="kg"><div><small>Collisions</small><b>'+S.length.toLocaleString()+'</b></div><div><small>Casualties</small><b>'+k+'</b></div><div><small>Fatal</small><b style="color:#ff4d5a">'+f+'</b></div><div><small>Serious</small><b style="color:#ffb020">'+s+'</b></div></div>'+
'<h3>What stands out</h3><div class="why">'+why+'</div>'+['sev','rt','jn','sp','li','inv','hr'].map(d=>'<h3>'+L[d]+'</h3>'+hb(countsS(S,d),d,null,false)).join('')+'<button class="go" id="go">See how it happens at street level</button>';
el._S=S}
function closeCard(){sel=null;$('#card').style.display='none'}
/* ---------- street-level replay ---------- */
const SIM={on:false,S:[],i:0,t0:0};
function rnd(s){s=s%2147483647||1;return()=>(s=s*16807%2147483647)/2147483647}
function veh(g,x,y,a,l,w,col){g.save();g.translate(x,y);g.rotate(a);g.shadowColor='#000';g.shadowBlur=8;g.fillStyle=col;g.beginPath();g.roundRect(-l/2,-w/2,l,w,w*.28);g.fill();g.shadowBlur=0;g.fillStyle='#0b1712cc';if(l>w*1.5)g.fillRect(l*.06,-w*.36,l*.2,w*.72);g.fillStyle='#fff3b0';g.fillRect(l/2-3,-w*.38,3,w*.22);g.fillRect(l/2-3,w*.16,3,w*.22);g.restore()}
function scene(g,w,h,t,i){const rg=rnd(i+7),rt=C.rt[i],jn=C.jn[i],inv=C.inv[i],sv=C.sev[i],spd=parseInt(NM.sp[C.sp[i]])||30,dk=C.li[i],we=C.we[i],ur=C.ur[i]==0;
const RW=Math.max(60,Math.min(h*.3,w*.2,120)),s=RW/90,cx=w/2,cy=h/2,rb=rt==0,du=rt==2,jc=!rb&&jn>=1&&jn<=5,tj=jn==1;
g.fillStyle=dk?'#07120d':'#173524';g.fillRect(0,0,w,h);
for(let n=0;n<34;n++){const x=rg()*w,y=rg()*h;if(Math.abs(y-cy)<RW*.95||((jc||rb)&&Math.abs(x-cx)<RW*.95))continue;if(ur){g.fillStyle=dk?'#16271e':'#2a4a38';g.fillRect(x,y,40*s+rg()*50*s,28*s+rg()*30*s)}else{g.fillStyle=dk?'#0d2417':'#1f6b3a';g.beginPath();g.arc(x,y,(9+rg()*10)*s,0,6.3);g.fill()}}
const as=dk?'#1c2622':'#2b3732';g.fillStyle=as;const rh=du?RW*1.35:RW;g.fillRect(0,cy-rh/2,w,rh);
if(jc||rb)g.fillRect(cx-RW/2,tj?cy:0,RW,tj?h-cy:h);
if(rt==4){g.save();g.translate(cx-RW,cy-RW*.5);g.rotate(.45);g.fillRect(0,0,RW*3,RW*.5);g.restore()}
if(rb){g.beginPath();g.arc(cx,cy,RW*.95,0,6.3);g.fill();g.fillStyle=dk?'#0d2417':'#1f6b3a';g.beginPath();g.arc(cx,cy,RW*.5,0,6.3);g.fill()}
if(du){g.fillStyle='#1f6b3a';g.fillRect(0,cy-RW*.07,w,RW*.14)}
g.strokeStyle='rgba(232,245,238,.7)';g.lineWidth=2;g.setLineDash([16*s,14*s]);if(!rb||true){g.beginPath();if(du){g.moveTo(0,cy-RW*.37);g.lineTo(w,cy-RW*.37);g.moveTo(0,cy+RW*.37);g.lineTo(w,cy+RW*.37)}else{g.moveTo(0,cy);g.lineTo(w,cy)}g.stroke()}g.setLineDash([]);
g.strokeStyle='rgba(255,255,255,.35)';g.beginPath();g.moveTo(0,cy-rh/2);g.lineTo(w,cy-rh/2);g.moveTo(0,cy+rh/2);g.lineTo(w,cy+rh/2);g.stroke();
const T0=Math.max(1.4,4.2-spd/22),tt=t%7,dt=Math.max(0,tt-T0),yA=cy+RW*(du?.45:.2);let P,A0,B0;
if(inv==0){P=[cx,yA];A0=[-70,yA];B0=[cx,cy-RW*.95]}else if(jc||rb){P=[cx+RW*.2,yA];A0=[-70,yA];B0=[cx+RW*.2,h+70]}else{P=[cx,cy+RW*.04];A0=[-70,yA];B0=[w+70,cy-RW*(du?.45:.2)]}
const dims=[[0,0],[22,6],[24,9],[54,26],[90,28],[44,22]][inv],cols=['#ffd24a','#ff9a3c','#ff9a3c','#e6e6e6','#ffb020','#e6e6e6'];
const place=(S0)=>{const f=Math.min(tt/T0,1),u=Math.atan2(P[1]-S0[1],P[0]-S0[0]),e=1-Math.exp(-dt*3);return[S0[0]+(P[0]-S0[0])*f+Math.cos(u)*e*30,S0[1]+(P[1]-S0[1])*f+Math.sin(u)*e*30,u]};
const a=place(A0),bb=place(B0);
if(dk){g.fillStyle='rgba(0,6,10,'+(dk==1?.4:.68)+')';g.fillRect(0,0,w,h);g.globalCompositeOperation='lighter';[a,bb].forEach(p=>{const fx=p[0]+Math.cos(p[2])*RW*.7,fy=p[1]+Math.sin(p[2])*RW*.7,gr=g.createRadialGradient(fx,fy,0,fx,fy,RW*.9);gr.addColorStop(0,'rgba(255,240,180,.4)');gr.addColorStop(1,'rgba(255,240,180,0)');g.fillStyle=gr;g.fillRect(fx-RW,fy-RW,RW*2,RW*2)});g.globalCompositeOperation='source-over'}
g.save();g.translate(a[0],a[1]);veh(g,0,0,a[2]-dt*.5*(1-Math.exp(-dt*3)),44*s,22*s,'#4aa3ff');g.restore();
if(inv==0){g.fillStyle=cols[0];g.shadowColor='#ffd24a';g.shadowBlur=10;g.beginPath();g.arc(bb[0],bb[1]+Math.sin(tt*9)*1.5,6*s,0,6.3);g.fill();g.shadowBlur=0}else veh(g,bb[0],bb[1],bb[2]+dt*.7*(1-Math.exp(-dt*3)),dims[0]*s,dims[1]*s,cols[inv]);
if(we==1||we==4){g.strokeStyle='rgba(180,220,255,.35)';g.lineWidth=1;g.beginPath();for(let n=0;n<110;n++){const x=(rg()*w+tt*60)%w,y=(rg()*h+tt*600)%h;g.moveTo(x,y);g.lineTo(x-4,y+14)}g.stroke()}
if(we==2||we==5){g.fillStyle='rgba(255,255,255,.7)';for(let n=0;n<90;n++){g.fillRect((rg()*w+tt*20)%w,(rg()*h+tt*90)%h,2.5,2.5)}}
if(we==6){g.fillStyle='rgba(205,228,218,.25)';g.fillRect(0,0,w,h)}
if(dt>0){const rr=Math.min(dt,2),al=Math.max(0,1-dt/2.2),gr=g.createRadialGradient(P[0],P[1],0,P[0],P[1],RW*(.4+rr*.7));gr.addColorStop(0,'rgba(255,77,90,'+al*.8+')');gr.addColorStop(1,'rgba(255,77,90,0)');g.fillStyle=gr;g.fillRect(P[0]-RW*2,P[1]-RW*2,RW*4,RW*4);
g.strokeStyle=SC[sv];g.globalAlpha=al;g.lineWidth=3;g.beginPath();g.arc(P[0],P[1],rr*RW*.7,0,6.3);g.stroke();for(let n=0;n<12;n++){const an=n*.52+rg(),l1=rr*RW*.5,l2=l1+10*s;g.beginPath();g.moveTo(P[0]+Math.cos(an)*l1,P[1]+Math.sin(an)*l1);g.lineTo(P[0]+Math.cos(an)*l2,P[1]+Math.sin(an)*l2);g.stroke()}g.globalAlpha=1;
const ty=P[1]-RW*.9,q=(16+Math.sin(tt*8)*2)*s;g.fillStyle='#ff4d5a';g.shadowColor='#ff4d5a';g.shadowBlur=18;g.beginPath();g.moveTo(P[0],ty-q);g.lineTo(P[0]+q,ty+q*.7);g.lineTo(P[0]-q,ty+q*.7);g.closePath();g.fill();g.shadowBlur=0;g.fillStyle='#fff';g.font='bold '+(16*s)+'px Segoe UI';g.textAlign='center';g.fillText('!',P[0],ty+q*.25);g.textAlign='left'}
g.fillStyle='#fff';g.strokeStyle='#e0202c';g.lineWidth=5;g.beginPath();g.arc(w-60,h-60,24,0,6.3);g.fill();g.stroke();g.fillStyle='#000';g.font='bold 18px Segoe UI';g.textAlign='center';g.fillText(spd,w-60,h-59);g.textAlign='left'}
function simInfo(){const i=SIM.S[SIM.i],row=(a,b)=>'<div class="l"><span>'+a+'</span><b>'+b+'</b></div>';
$('#si').innerHTML='<h2 style="color:'+SC[C.sev[i]]+'">'+NM.sev[C.sev[i]]+' collision</h2><small>'+(SIM.i+1)+' of '+SIM.S.length+' in this area, most severe first</small>'+
row('When',NM.dow[C.dow[i]]+' '+C.hr[i]+':00, '+NM.mo[C.mo[i]])+row('Road',NM.rt[C.rt[i]])+row('Junction',NM.jn[C.jn[i]])+row('Speed limit',NM.sp[C.sp[i]])+row('Light',NM.li[C.li[i]])+row('Weather',NM.we[C.we[i]])+row('Surface',NM.su[C.su[i]])+row('Setting',NM.ur[C.ur[i]])+row('Road user hit',NM.inv[C.inv[i]])+row('Vehicles / casualties',C.veh[i]+' / '+C.cas[i])+
'<p>Illustrative reconstruction: the road layout, speed limit, light, weather and road user come from the police record. The exact vehicle paths are not in the data.</p>'}
function openSim(S){SIM.S=S.slice().sort((a,b)=>C.sev[a]-C.sev[b]||((a*7919)%101)-((b*7919)%101)).slice(0,60);SIM.i=0;SIM.t0=performance.now();SIM.on=true;$('#sim').style.display='block';$('#card').style.display='none';simInfo()}
function step(d){SIM.i=(SIM.i+d+SIM.S.length)%SIM.S.length;SIM.t0=performance.now();simInfo()}
function loop(ts){const g=$('#ov').getContext('2d');g.setTransform(R,0,0,R,0,0);g.clearRect(0,0,W,H);
if(!SIM.on){HS.forEach((h,i)=>{const x=sx(h.x),y=sy(h.y);if(x<-20||x>W+20||y<-20||y>H+20)return;const p=(Math.sin(ts/380+i)+1)/2,r=8+Math.min(h.s/HS[0].s,1)*8;g.strokeStyle='rgba(255,77,90,'+(1-p)+')';g.lineWidth=2;g.beginPath();g.arc(x,y,r+p*14,0,6.3);g.stroke();g.fillStyle='#ff4d5a';g.shadowColor='#ff4d5a';g.shadowBlur=14;g.beginPath();g.arc(x,y,r*.55,0,6.3);g.fill();g.shadowBlur=0;g.fillStyle='#fff';g.font='bold 10px Segoe UI';g.textAlign='center';g.textBaseline='middle';g.fillText(i+1,x,y+.5)});g.textAlign='left';
if(sel){g.strokeStyle='#2dff8f';g.setLineDash([5,4]);g.lineWidth=1.5;g.beginPath();g.arc(sx(sel.x),sy(sel.y),Math.max(10,sel.r*V.k),0,6.3);g.stroke();g.setLineDash([])}}
else{const s=$('#sc').getContext('2d');s.setTransform(R,0,0,R,0,0);scene(s,W,H,(ts-SIM.t0)/1000,SIM.S[SIM.i])}
requestAnimationFrame(loop)}
/* ---------- events ---------- */
$('#nav').innerHTML=T.map((t,i)=>'<button data-t="'+i+'">'+t.n+'</button>').join('');
function nav(){document.querySelectorAll('#nav button').forEach((e,i)=>e.className=i==tab?'on':'')}
document.addEventListener('click',e=>{const t=e.target.closest('[data-t]');if(t){tab=+t.dataset.t;nav();update();return}
if(e.target.id==='rs'){F={};update();return}
if(e.target.id==='cx'){closeCard();return}
if(e.target.id==='go'){openSim($('#card')._S);return}
const h=e.target.closest('[data-h]');if(h){const o=HS[+h.dataset.h];V.cx=o.x;V.cy=o.y;V.k=Math.max(V.k,V.k0*8);req();hot();openCard(o.x,o.y,o.r*2);return}
const el=e.target.closest('.cl[data-d]');if(!el)return;const d=el.dataset.d,j=+el.dataset.j;
F[d]=F[d]||new Set();F[d].has(j)?F[d].delete(j):F[d].add(j);if(!F[d].size)delete F[d];update()});
const ov=$('#ov');
ov.addEventListener('wheel',e=>{e.preventDefault();const r=ov.getBoundingClientRect();zoomAt(e.deltaY<0?1.25:.8,e.clientX-r.left,e.clientY-r.top)},{passive:false});
ov.addEventListener('pointerdown',e=>{dr={x:e.clientX,y:e.clientY,cx:V.cx,cy:V.cy,m:0};ov.setPointerCapture(e.pointerId);ov.style.cursor='grabbing'});
ov.addEventListener('pointermove',e=>{if(!dr)return;const dx=e.clientX-dr.x,dy=e.clientY-dr.y;if(Math.hypot(dx,dy)>4)dr.m=1;if(dr.m){V.cx=dr.cx-dx/V.k;V.cy=dr.cy-dy/V.k;req()}});
ov.addEventListener('pointerup',e=>{ov.style.cursor='grab';const d=dr;dr=null;if(!d)return;if(d.m){hot();return}
const r=ov.getBoundingClientRect(),px=e.clientX-r.left,py=e.clientY-r.top;let best=null,bd=22;HS.forEach(h=>{const dd=Math.hypot(sx(h.x)-px,sy(h.y)-py);if(dd<bd){bd=dd;best=h}});
if(best)openCard(best.x,best.y,best.r*1.2);else openCard((px-W/2)/V.k+V.cx,(py-H/2)/V.k+V.cy,24/V.k)});
$('#zi').onclick=()=>zoomAt(1.5,W/2,H/2);$('#zo').onclick=()=>zoomAt(1/1.5,W/2,H/2);$('#zr').onclick=()=>{V.k=V.k0;V.cx=-3.4*KX;V.cy=-55.35;req();hot()};
$('#sx').onclick=()=>{SIM.on=false;$('#sim').style.display='none';if(sel)openCard(sel.x,sel.y,sel.r)};$('#sn').onclick=()=>step(1);$('#sp').onclick=()=>step(-1);
document.addEventListener('keydown',e=>{if(e.key==='Escape'){if(SIM.on)$('#sx').onclick();else closeCard()}});
addEventListener('resize',()=>{layout();hot()});nav();layout();update();requestAnimationFrame(loop);
</script></body></html>


In [ ]:
%%writefile land.json
[{"id":826,"c":[["-2.67,51.62 -2.74,51.58 -2.98,51.54 -3.08,51.5 -3.26,51.4 -3.29,51.39 -3.56,51.41 -3.76,51.54 -3.89,51.59 -3.94,51.6 -4,51.58 -4.12,51.57 -4.24,51.57 -4.17,51.63 -4.09,51.66 -4.28,51.68 -4.33,51.7 -4.39,51.74 -4.53,51.75 -4.6,51.74 -4.72,51.68 -4.9,51.63 -5.12,51.71 -5.17,51.74 -5.17,51.81 -5.2,51.86 -5.26,51.88 -5.18,51.95 -5.09,52 -4.88,52.04 -4.56,52.15 -4.38,52.2 -4.22,52.28 -4.15,52.33 -4.1,52.39 -4.05,52.47 -3.98,52.54 -4.05,52.56 -4.08,52.61 -4.07,52.66 -4.04,52.7 -4.07,52.76 -4.12,52.82 -4.11,52.87 -4.1,52.92 -4.23,52.91 -4.36,52.9 -4.47,52.86 -4.58,52.82 -4.68,52.81 -4.68,52.84 -4.64,52.89 -4.53,52.96 -4.4,53.01 -4.36,53.06 -4.33,53.1 -4.27,53.14 -4.11,53.22 -3.81,53.3 -3.76,53.31 -3.65,53.3 -3.53,53.31 -3.43,53.34 -3.32,53.35 -3.1,53.26 -3.17,53.39 -3.07,53.43 -2.92,53.3 -2.86,53.29 -2.75,53.31 -2.8,53.33 -2.85,53.33 -2.91,53.35 -2.97,53.39 -3.07,53.51 -3.06,53.59 -3,53.66 -2.93,53.73 -2.98,53.75 -3.03,53.77 -3.04,53.84 -3.03,53.91 -2.9,53.96 -2.86,54.04 -2.85,54.14 -2.87,54.18 -2.99,54.17 -3.05,54.15 -3.11,54.13 -3.17,54.13 -3.32,54.23 -3.41,54.31 -3.57,54.47 -3.59,54.56 -3.47,54.77 -3.27,54.91 -3.04,54.95 -3.08,54.96 -3.43,54.96 -3.55,54.95 -3.66,54.89 -3.72,54.88 -3.78,54.87 -3.84,54.84 -3.9,54.8 -3.96,54.78 -4.08,54.79 -4.13,54.78 -4.17,54.8 -4.21,54.84 -4.25,54.85 -4.3,54.84 -4.41,54.79 -4.52,54.76 -4.65,54.79 -4.82,54.85 -4.85,54.83 -4.89,54.77 -4.91,54.69 -5.03,54.76 -5.14,54.86 -5.17,54.92 -5.17,54.99 -5.12,55.01 -5.06,54.99 -4.97,55.15 -4.79,55.36 -4.72,55.42 -4.68,55.5 -4.69,55.55 -4.73,55.6 -4.89,55.7 -4.89,55.78 -4.87,55.87 -4.83,55.93 -4.81,55.94 -4.58,55.94 -4.67,55.97 -4.84,56.05 -4.84,56.08 -4.8,56.16 -4.82,56.15 -4.85,56.12 -4.93,56.03 -4.97,56.01 -5.09,55.99 -5.11,55.95 -5.14,55.93 -5.2,55.93 -5.21,55.89 -5.23,55.89 -5.25,55.93 -5.25,56 -5.22,56.07 -5.18,56.12 -5,56.23 -5.09,56.2 -5.28,56.09 -5.38,56.02 -5.41,56 -5.42,55.97 -5.42,55.95 -5.37,55.83 -5.39,55.77 -5.56,55.39 -5.59,55.35 -5.62,55.33 -5.65,55.33 -5.73,55.33 -5.77,55.36 -5.77,55.39 -5.75,55.44 -5.68,55.62 -5.65,55.67 -5.6,55.72 -5.51,55.8 -5.51,55.81 -5.57,55.79 -5.6,55.8 -5.62,55.81 -5.61,56.05 -5.56,56.13 -5.54,56.25 -5.49,56.35 -5.43,56.42 -5.39,56.51 -5.33,56.56 -5.31,56.62 -5.24,56.69 -5.19,56.76 -5.22,56.75 -5.56,56.57 -5.65,56.53 -5.77,56.54 -5.87,56.56 -5.94,56.61 -5.97,56.69 -6.06,56.69 -6.13,56.71 -6.13,56.72 -6.04,56.76 -5.88,56.78 -5.73,56.85 -5.86,56.9 -5.85,56.92 -5.74,56.96 -5.59,57.1 -5.56,57.23 -5.63,57.29 -5.66,57.33 -5.79,57.38 -5.82,57.44 -5.8,57.47 -5.76,57.5 -5.69,57.52 -5.58,57.55 -5.68,57.57 -5.72,57.6 -5.74,57.64 -5.74,57.67 -5.69,57.78 -5.66,57.82 -5.61,57.88 -5.35,57.88 -5.32,57.9 -5.29,57.91 -5.16,57.88 -5.18,57.91 -5.39,58.04 -5.41,58.07 -5.35,58.14 -5.35,58.18 -5.36,58.21 -5.34,58.24 -5.27,58.25 -5.06,58.25 -5.01,58.26 -5.03,58.3 -5.08,58.35 -5.09,58.38 -5.08,58.42 -5.07,58.49 -5.07,58.52 -5.02,58.57 -4.98,58.58 -4.92,58.59 -4.81,58.57 -4.76,58.55 -4.71,58.51 -4.68,58.51 -4.53,58.56 -4.49,58.57 -4.43,58.51 -4.19,58.56 -3.86,58.58 -3.66,58.61 -3.45,58.62 -3.26,58.65 -3.05,58.63 -3.05,58.62 -3.06,58.59 -3.11,58.52 -3.1,58.43 -3.11,58.41 -3.14,58.38 -3.21,58.32 -3.41,58.24 -3.77,58.05 -3.99,57.96 -4.02,57.91 -4.03,57.85 -3.91,57.84 -3.86,57.82 -3.89,57.79 -4.08,57.68 -4.13,57.58 -3.99,57.58 -3.87,57.6 -3.63,57.66 -3.4,57.71 -3.3,57.71 -3.08,57.67 -3.04,57.67 -2.95,57.69 -2.86,57.69 -2.24,57.68 -2.08,57.7 -1.96,57.68 -1.87,57.61 -1.78,57.49 -1.78,57.47 -1.83,57.42 -1.94,57.35 -2.02,57.26 -2.05,57.21 -2.06,57.15 -2.09,57.1 -2.26,56.86 -2.43,56.73 -2.5,56.64 -2.59,56.56 -2.68,56.51 -2.77,56.48 -3.05,56.45 -3.12,56.43 -3.21,56.38 -3.31,56.36 -3.2,56.37 -3.09,56.39 -2.89,56.4 -2.65,56.32 -2.67,56.25 -2.77,56.2 -2.98,56.19 -3.18,56.08 -3.27,56.05 -3.36,56.03 -3.48,56.03 -3.7,56.06 -3.79,56.09 -3.7,56.04 -3.61,56.02 -3.05,55.95 -3.02,55.96 -2.84,56.03 -2.6,56.03 -2.15,55.9 -2.02,55.81 -1.83,55.67 -1.73,55.62 -1.65,55.57 -1.61,55.5 -1.52,55.26 -1.42,55.03 -1.29,54.77 -1.23,54.7 -1.15,54.66 -0.76,54.54 -0.67,54.5 -0.52,54.39 -0.37,54.28 -0.23,54.19 -0.08,54.12 -0.16,54.08 -0.21,54.02 -0.17,53.94 -0.11,53.87 0.01,53.74 0.12,53.61 0.08,53.63 0.04,53.64 -0.02,53.64 -0.07,53.64 -0.17,53.68 -0.27,53.74 -0.46,53.72 -0.57,53.72 -0.66,53.72 -0.48,53.7 -0.29,53.69 0.13,53.47 0.27,53.34 0.35,53.16 0.3,53.08 0.21,53.03 0.12,52.97 0.05,52.91 0.28,52.81 0.33,52.81 0.38,52.83 0.43,52.86 0.52,52.94 0.56,52.97 0.7,52.98 0.83,52.97 0.95,52.95 1.06,52.96 1.27,52.92 1.38,52.89 1.66,52.75 1.72,52.68 1.74,52.58 1.75,52.47 1.7,52.37 1.65,52.28 1.61,52.16 1.59,52.12 1.56,52.09 1.41,51.99 1.32,51.96 1.28,51.97 1.23,51.97 1.23,51.95 1.27,51.9 1.28,51.84 1.19,51.8 1.1,51.79 0.96,51.81 0.75,51.73 0.9,51.69 0.93,51.65 0.89,51.57 0.8,51.54 0.7,51.52 0.59,51.52 0.51,51.5 0.42,51.47 0.53,51.49 0.6,51.47 0.65,51.41 0.69,51.39 0.89,51.36 1.01,51.36 1.26,51.38 1.37,51.37 1.42,51.36 1.42,51.31 1.4,51.18 1.37,51.16 1.05,51.05 0.98,50.97 0.96,50.93 0.77,50.93 0.69,50.89 0.53,50.85 0.42,50.82 0.3,50.78 0.2,50.76 -0.2,50.82 -0.45,50.81 -0.79,50.76 -0.87,50.77 -1,50.82 -1.13,50.84 -1.28,50.86 -1.42,50.9 -1.33,50.82 -1.52,50.75 -1.6,50.73 -1.69,50.74 -1.87,50.71 -2.03,50.72 -2.01,50.67 -1.96,50.63 -2,50.61 -2.04,50.6 -2.35,50.64 -2.4,50.63 -2.43,50.6 -2.55,50.62 -2.66,50.67 -2.78,50.71 -2.9,50.72 -3,50.72 -3.4,50.63 -3.49,50.55 -3.53,50.43 -3.58,50.32 -3.68,50.24 -3.79,50.23 -3.9,50.29 -4.1,50.35 -4.17,50.39 -4.2,50.39 -4.22,50.38 -4.3,50.36 -4.38,50.36 -4.51,50.34 -4.73,50.29 -4.82,50.26 -5.01,50.16 -5.05,50.13 -5.12,50.04 -5.23,50.02 -5.32,50.08 -5.43,50.11 -5.55,50.08 -5.62,50.05 -5.65,50.08 -5.66,50.13 -5.57,50.2 -5.34,50.25 -5.14,50.37 -5.04,50.45 -5.01,50.5 -4.96,50.52 -4.89,50.53 -4.86,50.58 -4.58,50.78 -4.56,50.82 -4.55,50.9 -4.52,50.98 -4.3,51.03 -4.19,51.19 -4.16,51.2 -3.84,51.23 -3.61,51.23 -3.38,51.2 -3.26,51.19 -3.14,51.21 -3.04,51.25 -2.88,51.41 -2.79,51.47 -2.69,51.54 -2.59,51.61 -2.43,51.74 -2.54,51.7 -2.67,51.62"],["-4.2,53.32 -4.16,53.3 -4.05,53.31 -4.08,53.26 -4.2,53.22 -4.28,53.17 -4.37,53.13 -4.42,53.18 -4.47,53.18 -4.55,53.26 -4.57,53.39 -4.46,53.42 -4.31,53.42 -4.2,53.32"],["-2.55,59.23 -2.66,59.23 -2.6,59.29 -2.54,59.3 -2.41,59.3 -2.43,59.27 -2.55,59.23"],["-1.04,60.51 -1.07,60.5 -1.16,60.6 -1.09,60.72 -1.01,60.72 -0.99,60.69 -1,60.66 -1.05,60.65 -1.05,60.65 -1.04,60.59 -1.04,60.53 -1.04,60.51"],["-1.31,60.54 -1.29,60.47 -1.24,60.48 -1.16,60.42 -1.12,60.42 -1.05,60.44 -1.07,60.38 -1.13,60.21 -1.15,60.18 -1.16,60.12 -1.18,60.11 -1.2,60.01 -1.24,59.97 -1.28,59.89 -1.3,59.88 -1.36,59.91 -1.3,60.04 -1.28,60.11 -1.29,60.15 -1.32,60.19 -1.41,60.19 -1.48,60.17 -1.5,60.19 -1.5,60.22 -1.52,60.23 -1.61,60.23 -1.64,60.24 -1.66,60.26 -1.67,60.28 -1.58,60.3 -1.5,60.29 -1.37,60.33 -1.45,60.47 -1.55,60.48 -1.57,60.5 -1.55,60.52 -1.5,60.53 -1.41,60.6 -1.36,60.61 -1.3,60.61 -1.31,60.54"],["-0.78,60.81 -0.78,60.8 -0.83,60.72 -0.83,60.68 -0.91,60.69 -0.92,60.7 -0.94,60.75 -0.93,60.8 -0.92,60.81 -0.89,60.82 -0.87,60.81 -0.82,60.83 -0.8,60.83 -0.78,60.81"],["-3.17,58.79 -3.22,58.78 -3.28,58.78 -3.37,58.84 -3.4,58.88 -3.39,58.91 -3.36,58.92 -3.27,58.9 -3.23,58.86 -3.22,58.83 -3.21,58.81 -3.16,58.8 -3.17,58.79"],["-2.93,58.74 -2.94,58.74 -2.98,58.76 -3.04,58.82 -2.94,58.84 -2.9,58.83 -2.91,58.8 -2.93,58.74"],["-3.06,59.03 -3.07,59.01 -2.99,59.01 -2.89,58.98 -2.82,58.98 -2.76,58.96 -2.79,58.91 -2.83,58.89 -2.86,58.89 -2.99,58.94 -3.17,58.92 -3.2,58.93 -3.22,58.94 -3.23,58.96 -3.23,58.99 -3.24,59 -3.3,58.97 -3.33,58.97 -3.35,58.99 -3.35,59.02 -3.35,59.06 -3.31,59.13 -3.25,59.14 -3.16,59.14 -3.05,59.1 -3.02,59.08 -3.02,59.06 -3.06,59.03"],["-2.73,59.19 -2.82,59.16 -2.85,59.18 -2.86,59.25 -2.96,59.27 -3.02,59.29 -3.05,59.32 -3.04,59.33 -2.98,59.35 -2.86,59.29 -2.81,59.24 -2.73,59.23 -2.72,59.22 -2.73,59.19"],["-6.61,56.59 -6.67,56.58 -6.67,56.59 -6.57,56.66 -6.51,56.67 -6.49,56.67 -6.53,56.63 -6.61,56.59"],["-5.11,55.45 -5.23,55.45 -5.28,55.46 -5.33,55.48 -5.39,55.62 -5.37,55.67 -5.34,55.69 -5.32,55.71 -5.25,55.72 -5.19,55.69 -5.16,55.67 -5.11,55.57 -5.1,55.49 -5.11,55.45"],["-5.78,56.34 -6.18,56.29 -6.31,56.29 -6.33,56.32 -6.3,56.34 -6.19,56.36 -6.14,56.49 -6.31,56.55 -6.32,56.57 -6.31,56.6 -6.29,56.61 -6.18,56.64 -6.14,56.65 -6.1,56.65 -6.03,56.61 -5.95,56.54 -5.84,56.52 -5.76,56.49 -5.78,56.34"],["-6.13,55.93 -6.09,55.8 -6.06,55.72 -6.06,55.7 -6.09,55.66 -6.25,55.61 -6.31,55.61 -6.31,55.62 -6.27,55.67 -6.3,55.73 -6.29,55.77 -6.3,55.78 -6.33,55.77 -6.45,55.7 -6.49,55.7 -6.5,55.71 -6.47,55.77 -6.46,55.81 -6.45,55.83 -6.41,55.85 -6.37,55.87 -6.35,55.87 -6.31,55.86 -6.22,55.91 -6.13,55.93"],["-5.97,55.81 -5.99,55.8 -6.04,55.81 -6.06,55.82 -6.07,55.85 -6.07,55.89 -6.04,55.93 -5.91,55.97 -5.97,55.99 -5.97,56 -5.94,56.05 -5.8,56.11 -5.76,56.12 -5.73,56.12 -5.8,56.01 -5.97,55.81"],["-6.2,58.36 -6.33,58.19 -6.37,58.18 -6.42,58.14 -6.55,58.09 -6.44,58.09 -6.4,58.08 -6.4,58.04 -6.42,58.02 -6.58,57.94 -6.68,57.91 -6.8,57.83 -6.85,57.83 -6.91,57.77 -6.96,57.75 -6.98,57.75 -7.01,57.76 -7.08,57.81 -6.96,57.87 -6.94,57.89 -6.86,57.92 -6.86,57.93 -7,57.97 -7.06,58 -7.05,58.02 -6.99,58.05 -7.02,58.05 -7.04,58.07 -7.08,58.08 -7.09,58.1 -7.09,58.14 -7.09,58.18 -7.04,58.2 -7.03,58.22 -7.01,58.23 -6.95,58.22 -6.89,58.18 -6.81,58.2 -6.73,58.19 -6.72,58.2 -6.79,58.28 -6.78,58.3 -6.74,58.32 -6.54,58.38 -6.3,58.49 -6.24,58.5 -6.22,58.49 -6.19,58.43 -6.2,58.36"],["-6.28,56.96 -6.31,56.95 -6.35,56.95 -6.38,56.97 -6.43,57.02 -6.32,57.05 -6.28,57.03 -6.26,57.01 -6.26,56.98 -6.28,56.96"],["-6.14,57.51 -6.15,57.46 -6.17,57.41 -6.14,57.35 -6.14,57.31 -6.09,57.3 -6.07,57.28 -5.88,57.26 -5.7,57.27 -5.67,57.25 -5.67,57.23 -5.7,57.2 -5.79,57.15 -5.91,57.06 -5.95,57.05 -5.99,57.04 -6.01,57.05 -6.04,57.2 -6.16,57.18 -6.27,57.18 -6.32,57.2 -6.36,57.24 -6.44,57.33 -6.68,57.36 -6.74,57.41 -6.76,57.44 -6.75,57.46 -6.71,57.5 -6.64,57.48 -6.6,57.49 -6.58,57.51 -6.58,57.52 -6.62,57.55 -6.62,57.56 -6.38,57.6 -6.36,57.67 -6.31,57.67 -6.25,57.65 -6.17,57.59 -6.14,57.51"],["-7.21,57.68 -7.09,57.63 -7.18,57.53 -7.32,57.53 -7.52,57.6 -7.52,57.62 -7.5,57.64 -7.47,57.65 -7.44,57.66 -7.39,57.64 -7.32,57.66 -7.27,57.66 -7.21,57.68"],["-7.25,57.12 -7.29,57.11 -7.35,57.12 -7.38,57.13 -7.41,57.19 -7.42,57.23 -7.41,57.3 -7.41,57.38 -7.3,57.38 -7.27,57.37 -7.25,57.13 -7.25,57.12"],["-7.42,56.97 -7.5,56.95 -7.54,56.96 -7.54,56.97 -7.52,57.01 -7.45,57.02 -7.41,57 -7.4,56.98 -7.42,56.97"],["-6.22,54.09 -6.3,54.09 -6.36,54.08 -6.4,54.06 -6.44,54.06 -6.55,54.06 -6.65,54.06 -6.67,54.08 -6.65,54.16 -6.67,54.18 -6.77,54.2 -6.8,54.21 -6.86,54.27 -6.87,54.29 -6.88,54.33 -6.94,54.37 -7.01,54.41 -7.05,54.41 -7.13,54.35 -7.2,54.3 -7.18,54.28 -7.16,54.24 -7.19,54.21 -7.31,54.16 -7.32,54.13 -7.36,54.12 -7.41,54.14 -7.54,54.13 -7.61,54.14 -7.68,54.19 -7.85,54.22 -7.89,54.28 -7.92,54.3 -8.12,54.41 -8.15,54.45 -8.12,54.48 -8.04,54.51 -7.79,54.57 -7.75,54.59 -7.75,54.62 -7.82,54.64 -7.89,54.67 -7.91,54.68 -7.91,54.7 -7.87,54.72 -7.8,54.72 -7.74,54.71 -7.69,54.73 -7.61,54.75 -7.55,54.77 -7.5,54.83 -7.45,54.88 -7.45,54.91 -7.4,55 -7.38,55.03 -7.22,55.09 -7.18,55.06 -7.1,55.05 -7.03,55.08 -6.95,55.18 -6.89,55.19 -6.82,55.18 -6.7,55.19 -6.47,55.24 -6.37,55.24 -6.23,55.22 -6.13,55.22 -6.04,55.14 -5.99,55.03 -5.87,54.92 -5.72,54.82 -5.71,54.76 -5.77,54.72 -5.88,54.68 -5.88,54.64 -5.81,54.66 -5.74,54.67 -5.58,54.66 -5.53,54.62 -5.49,54.55 -5.47,54.5 -5.48,54.44 -5.52,54.46 -5.57,54.51 -5.61,54.54 -5.67,54.55 -5.65,54.48 -5.66,54.38 -5.63,54.37 -5.56,54.37 -5.61,54.27 -5.71,54.25 -5.83,54.24 -5.86,54.2 -5.88,54.16 -5.94,54.09 -6.02,54.05 -6.12,54.06 -6.22,54.09"],["-1.06,50.69 -1.15,50.66 -1.18,50.62 -1.2,50.6 -1.25,50.59 -1.31,50.59 -1.51,50.67 -1.56,50.67 -1.52,50.7 -1.38,50.73 -1.31,50.77 -1.14,50.74 -1.06,50.69"]]},{"id":528,"c":[["5.99,50.75 5.89,50.75 5.8,50.75 5.75,50.76 5.69,50.78 5.67,50.81 5.64,50.84 5.65,50.87 5.74,50.93 5.75,50.95 5.74,50.96 5.75,50.99 5.82,51.09 5.83,51.13 5.8,51.15 5.75,51.17 5.61,51.2 5.54,51.24 5.51,51.28 5.48,51.29 5.43,51.27 5.31,51.26 5.21,51.28 5.1,51.35 5.07,51.41 5.06,51.45 5.03,51.47 4.99,51.45 4.94,51.41 4.85,51.4 4.82,51.41 4.82,51.43 4.81,51.45 4.78,51.48 4.76,51.49 4.64,51.42 4.59,51.42 4.53,51.45 4.5,51.47 4.44,51.46 4.38,51.43 4.4,51.37 4.37,51.36 4.3,51.36 4.22,51.39 4.14,51.4 4.01,51.44 3.82,51.41 3.69,51.45 3.59,51.45 3.52,51.49 3.45,51.54 3.5,51.58 3.55,51.59 3.74,51.6 3.89,51.57 4.14,51.46 4.21,51.46 4.28,51.47 4.24,51.5 4.17,51.52 4.08,51.55 4.01,51.6 4.18,51.61 4.16,51.63 4.13,51.67 3.95,51.81 3.98,51.85 4.03,51.93 4.08,51.99 4.13,52.01 4.21,52.06 4.38,52.2 4.48,52.31 4.56,52.44 4.68,52.81 4.71,52.87 4.77,52.94 4.84,52.93 4.89,52.91 5.06,52.96 5.36,53.1 5.45,53.21 5.53,53.27 5.87,53.38 6.06,53.41 6.35,53.42 6.56,53.43 6.82,53.44 6.91,53.38 6.97,53.33 7.06,53.3 7.2,53.28 7.19,53.19 7.19,53 7.18,52.97 7.12,52.89 7.05,52.75 7.03,52.65 7.01,52.63 6.75,52.63 6.71,52.62 6.71,52.6 6.72,52.57 6.71,52.55 6.69,52.53 6.7,52.5 6.75,52.46 6.83,52.44 6.92,52.44 6.97,52.44 7,52.42 7.04,52.38 7.03,52.33 7.02,52.27 6.98,52.21 6.86,52.14 6.8,52.11 6.75,52.1 6.72,52.08 6.71,52.06 6.72,52.04 6.8,51.98 6.8,51.97 6.77,51.94 6.74,51.91 6.52,51.85 6.42,51.86 6.37,51.83 6.36,51.82 6.3,51.85 6.17,51.88 6.12,51.87 6.09,51.85 6.01,51.83 5.95,51.8 5.95,51.76 6.05,51.66 6.09,51.64 6.09,51.6 6.14,51.55 6.19,51.49 6.2,51.45 6.19,51.41 6.17,51.36 6.08,51.22 6.08,51.2 6.08,51.18 6.11,51.17 6.14,51.16 6.13,51.15 5.96,51.06 5.94,51.04 5.87,51.05 5.86,51.03 5.87,51.01 5.9,50.98 5.96,50.97 6.01,50.95 6.05,50.91 5.99,50.75"],["4.22,51.39 4.21,51.35 4.17,51.31 4.04,51.25 3.9,51.21 3.83,51.21 3.78,51.23 3.76,51.25 3.68,51.28 3.58,51.29 3.52,51.26 3.47,51.24 3.43,51.25 3.4,51.26 3.38,51.29 3.35,51.38 3.43,51.39 3.59,51.4 3.72,51.37 3.88,51.36 4.01,51.4 4.11,51.36 4.22,51.39"],["3.95,51.74 4.05,51.68 4.07,51.67 4.07,51.65 3.95,51.63 3.82,51.69 3.73,51.68 3.7,51.71 3.7,51.73 3.79,51.75 3.95,51.74"]]},{"id":372,"c":[["-9.95,53.91 -9.95,53.88 -10.03,53.92 -10.06,53.96 -10.27,53.98 -10.18,54.02 -10.14,54.01 -10,54 -9.96,53.99 -9.95,53.91"],["-7.22,55.09 -7.38,55.03 -7.4,55 -7.45,54.91 -7.45,54.88 -7.5,54.83 -7.55,54.77 -7.61,54.75 -7.69,54.73 -7.74,54.71 -7.8,54.72 -7.87,54.72 -7.91,54.7 -7.91,54.68 -7.89,54.67 -7.82,54.64 -7.75,54.62 -7.75,54.59 -7.79,54.57 -8.04,54.51 -8.12,54.48 -8.15,54.45 -8.12,54.41 -7.92,54.3 -7.89,54.28 -7.85,54.22 -7.68,54.19 -7.61,54.14 -7.54,54.13 -7.41,54.14 -7.36,54.12 -7.32,54.13 -7.31,54.16 -7.19,54.21 -7.16,54.24 -7.18,54.28 -7.2,54.3 -7.13,54.35 -7.05,54.41 -7.01,54.41 -6.94,54.37 -6.88,54.33 -6.87,54.29 -6.86,54.27 -6.8,54.21 -6.77,54.2 -6.67,54.18 -6.65,54.16 -6.67,54.08 -6.65,54.06 -6.55,54.06 -6.44,54.06 -6.4,54.06 -6.36,54.08 -6.3,54.09 -6.22,54.09 -6.18,54.05 -6.16,54.02 -6.23,54 -6.31,54.01 -6.35,53.99 -6.35,53.94 -6.32,53.88 -6.27,53.84 -6.23,53.75 -6.19,53.64 -6.14,53.58 -6.13,53.5 -6.14,53.46 -6.13,53.39 -6.15,53.37 -6.14,53.3 -6.07,53.17 -6.05,53.09 -6.03,52.93 -6.07,52.87 -6.13,52.81 -6.17,52.74 -6.2,52.66 -6.22,52.54 -6.35,52.4 -6.4,52.37 -6.46,52.34 -6.32,52.25 -6.44,52.2 -6.56,52.19 -6.7,52.21 -6.78,52.21 -6.86,52.18 -6.89,52.16 -6.91,52.17 -6.96,52.25 -7,52.17 -7.08,52.14 -7.22,52.14 -7.44,52.12 -7.53,52.1 -7.56,52.06 -7.59,52.02 -7.63,51.99 -7.67,51.98 -7.84,51.95 -7.87,51.93 -7.95,51.87 -8.06,51.83 -8.15,51.81 -8.22,51.85 -8.25,51.88 -8.29,51.89 -8.41,51.89 -8.37,51.88 -8.35,51.85 -8.34,51.79 -8.35,51.74 -8.41,51.71 -8.48,51.71 -8.59,51.65 -8.74,51.64 -8.81,51.58 -9.3,51.5 -9.32,51.5 -9.39,51.52 -9.46,51.53 -9.53,51.52 -9.74,51.47 -9.84,51.48 -9.71,51.6 -9.54,51.66 -9.52,51.68 -9.58,51.69 -9.9,51.65 -10.01,51.61 -10.12,51.6 -10.07,51.66 -9.93,51.73 -9.85,51.77 -9.8,51.78 -9.75,51.82 -9.6,51.87 -10.09,51.77 -10.21,51.78 -10.24,51.81 -10.34,51.8 -10.38,51.87 -10.23,51.97 -10.15,52.02 -10.04,52.04 -9.95,52.08 -9.91,52.12 -9.96,52.14 -10.25,52.13 -10.39,52.13 -10.38,52.17 -10.36,52.21 -10.21,52.27 -10.13,52.28 -10.06,52.28 -9.99,52.26 -9.94,52.24 -9.77,52.25 -9.84,52.29 -9.85,52.38 -9.91,52.4 -9.84,52.44 -9.76,52.47 -9.63,52.55 -9.59,52.56 -9.33,52.58 -9.06,52.62 -8.78,52.68 -8.92,52.71 -8.99,52.76 -9.1,52.67 -9.17,52.63 -9.39,52.62 -9.46,52.63 -9.56,52.65 -9.59,52.64 -9.62,52.62 -9.77,52.58 -9.92,52.57 -9.74,52.65 -9.52,52.78 -9.47,52.82 -9.39,52.9 -9.42,52.93 -9.46,52.95 -9.3,53.1 -9.24,53.12 -9.14,53.13 -9.06,53.15 -9.03,53.15 -9,53.16 -8.93,53.21 -9.03,53.24 -9.14,53.25 -9.47,53.24 -9.51,53.24 -9.56,53.25 -9.58,53.27 -9.6,53.32 -9.62,53.33 -9.7,53.33 -9.77,53.32 -9.83,53.32 -9.88,53.34 -9.79,53.39 -9.9,53.41 -10,53.4 -10.09,53.41 -10.09,53.45 -10.05,53.48 -10.11,53.51 -10.12,53.55 -10.06,53.57 -10,53.56 -9.88,53.59 -9.72,53.6 -9.86,53.63 -9.91,53.66 -9.91,53.7 -9.9,53.73 -9.74,53.78 -9.58,53.8 -9.59,53.84 -9.58,53.88 -9.75,53.89 -9.91,53.86 -9.89,53.94 -9.86,54 -9.85,54.05 -9.86,54.09 -9.93,54.08 -9.95,54.14 -9.98,54.19 -10.09,54.16 -10.09,54.22 -10.06,54.26 -10,54.28 -9.93,54.27 -9.83,54.27 -9.72,54.3 -9.56,54.31 -9.32,54.3 -9.15,54.21 -9.1,54.22 -9.03,54.28 -9,54.29 -8.75,54.26 -8.59,54.23 -8.54,54.24 -8.57,54.3 -8.62,54.35 -8.56,54.4 -8.47,54.44 -8.42,54.46 -8.29,54.49 -8.23,54.51 -8.19,54.58 -8.13,54.64 -8.46,54.61 -8.76,54.68 -8.71,54.73 -8.65,54.76 -8.54,54.78 -8.53,54.81 -8.47,54.83 -8.38,54.89 -8.41,54.96 -8.39,55.02 -8.33,55.06 -8.3,55.11 -8.27,55.15 -8.14,55.16 -8,55.2 -7.96,55.19 -7.8,55.2 -7.75,55.19 -7.76,55.25 -7.67,55.26 -7.63,55.24 -7.61,55.2 -7.57,55.17 -7.56,55.12 -7.59,55.08 -7.63,55.05 -7.59,55.02 -7.66,54.97 -7.58,54.99 -7.48,55.05 -7.48,55.09 -7.5,55.14 -7.53,55.19 -7.52,55.25 -7.46,55.28 -7.3,55.3 -7.37,55.36 -7.31,55.37 -7.25,55.35 -7.16,55.3 -7.06,55.27 -6.96,55.24 -7.06,55.18 -7.17,55.14 -7.22,55.09"]]},{"id":250,"c":[["7.62,47.59 7.49,47.55 7.47,47.51 7.42,47.46 7.34,47.43 7.27,47.43 7.2,47.43 7.17,47.45 7.17,47.47 7.14,47.49 7.05,47.49 6.97,47.45 6.9,47.39 6.92,47.36 6.99,47.35 7,47.34 7,47.32 6.98,47.3 6.95,47.27 6.82,47.16 6.69,47.06 6.67,47.03 6.63,47 6.46,46.95 6.44,46.93 6.43,46.83 6.41,46.75 6.28,46.68 6.16,46.61 6.13,46.57 6.11,46.52 6.07,46.46 6.06,46.43 6.12,46.38 6.11,46.34 6.1,46.28 6.04,46.24 5.97,46.21 5.97,46.15 6.01,46.14 6.09,46.15 6.2,46.19 6.27,46.25 6.23,46.31 6.22,46.32 6.23,46.33 6.32,46.39 6.43,46.43 6.58,46.44 6.76,46.42 6.78,46.41 6.77,46.37 6.78,46.31 6.82,46.28 6.77,46.16 6.81,46.13 6.86,46.09 6.9,46.05 6.95,46.02 7,45.96 7.02,45.93 6.94,45.87 6.81,45.81 6.79,45.78 6.79,45.74 6.81,45.71 6.88,45.67 6.96,45.58 7.01,45.5 7.13,45.42 7.16,45.4 7.15,45.38 7.12,45.35 7.08,45.24 7.03,45.22 6.98,45.21 6.84,45.13 6.78,45.15 6.69,45.14 6.63,45.12 6.63,45.07 6.69,45.02 6.72,44.97 6.74,44.92 6.8,44.88 6.89,44.86 6.94,44.86 6.97,44.85 6.99,44.83 7.03,44.72 7.01,44.69 6.96,44.68 6.93,44.63 6.87,44.56 6.84,44.51 6.88,44.46 6.89,44.43 6.87,44.39 6.9,44.34 6.97,44.28 7.15,44.2 7.32,44.14 7.37,44.13 7.6,44.17 7.64,44.16 7.67,44.12 7.68,44.08 7.65,44.03 7.59,43.96 7.52,43.91 7.48,43.86 7.49,43.82 7.49,43.77 7.44,43.75 7.44,43.76 7.41,43.77 7.4,43.77 7.38,43.75 7.38,43.73 7.26,43.7 7.18,43.66 6.86,43.44 6.72,43.37 6.69,43.33 6.66,43.26 6.57,43.2 6.49,43.17 6.31,43.14 6.11,43.07 6.03,43.1 5.81,43.1 5.67,43.18 5.41,43.23 5.32,43.35 5.2,43.35 5.12,43.35 5.07,43.37 5.06,43.41 5.06,43.44 4.98,43.43 4.91,43.43 4.87,43.41 4.84,43.39 4.81,43.41 4.79,43.4 4.79,43.38 4.71,43.37 4.63,43.39 4.41,43.45 4.38,43.46 4.22,43.48 4.16,43.5 4.11,43.56 4.07,43.58 4.05,43.59 3.91,43.56 3.86,43.52 3.79,43.46 3.26,43.19 3.16,43.08 3.05,42.91 3.04,42.84 3.09,42.59 3.2,42.46 3.21,42.43 3.15,42.43 3.05,42.45 2.97,42.47 2.89,42.46 2.82,42.43 2.75,42.41 2.7,42.41 2.67,42.39 2.66,42.36 2.65,42.34 2.57,42.35 2.37,42.39 2.2,42.42 2.1,42.39 2.03,42.35 1.99,42.36 1.95,42.39 1.93,42.43 1.86,42.46 1.7,42.5 1.72,42.53 1.74,42.56 1.74,42.58 1.71,42.6 1.57,42.64 1.5,42.64 1.46,42.62 1.43,42.6 1.35,42.69 1.29,42.71 1.21,42.71 1.11,42.74 1.01,42.78 0.77,42.84 0.7,42.85 0.67,42.84 0.65,42.8 0.64,42.7 0.63,42.69 0.52,42.69 0.38,42.7 0.31,42.69 0.25,42.69 0.2,42.72 -0.04,42.69 -0.08,42.7 -0.14,42.75 -0.21,42.79 -0.26,42.8 -0.3,42.83 -0.34,42.83 -0.4,42.81 -0.48,42.8 -0.55,42.8 -0.59,42.8 -0.74,42.91 -0.76,42.94 -0.84,42.95 -0.93,42.95 -1.18,43.02 -1.29,43.06 -1.3,43.08 -1.3,43.1 -1.32,43.1 -1.35,43.06 -1.37,43.04 -1.4,43.03 -1.43,43.04 -1.46,43.05 -1.48,43.07 -1.46,43.11 -1.42,43.15 -1.41,43.2 -1.41,43.24 -1.47,43.27 -1.56,43.28 -1.63,43.28 -1.71,43.31 -1.76,43.32 -1.79,43.37 -1.79,43.41 -1.63,43.44 -1.49,43.56 -1.34,44.02 -1.24,44.56 -1.17,44.66 -1.08,44.69 -1.15,44.76 -1.2,44.73 -1.22,44.69 -1.24,44.67 -1.19,45.16 -1.15,45.34 -1.08,45.53 -0.94,45.46 -0.83,45.38 -0.77,45.31 -0.69,45.09 -0.64,45.05 -0.55,45 -0.58,45.05 -0.64,45.09 -0.73,45.38 -0.79,45.47 -0.88,45.54 -1.17,45.69 -1.2,45.71 -1.21,45.77 -1.11,45.77 -1.03,45.74 -1.04,45.77 -1.07,45.81 -1.1,45.92 -1.14,46.2 -1.13,46.25 -1.15,46.31 -1.24,46.32 -1.31,46.33 -1.39,46.35 -1.79,46.52 -1.92,46.69 -2.06,46.81 -2.09,46.87 -2.09,46.92 -2.02,47.04 -2.08,47.11 -2.14,47.13 -2.2,47.16 -2.15,47.22 -2.11,47.26 -2.03,47.27 -1.92,47.26 -1.82,47.23 -1.74,47.22 -1.97,47.31 -2.35,47.28 -2.44,47.29 -2.5,47.31 -2.53,47.38 -2.48,47.41 -2.43,47.47 -2.48,47.51 -2.55,47.53 -2.67,47.53 -2.77,47.51 -2.8,47.54 -2.73,47.6 -2.79,47.63 -2.86,47.61 -2.96,47.6 -3.07,47.62 -3.16,47.69 -3.22,47.69 -3.26,47.69 -3.33,47.71 -3.4,47.72 -3.44,47.71 -3.51,47.75 -3.9,47.84 -4.07,47.85 -4.22,47.81 -4.31,47.82 -4.38,47.88 -4.43,47.97 -4.68,48.04 -4.63,48.09 -4.51,48.1 -4.38,48.13 -4.33,48.17 -4.43,48.22 -4.51,48.23 -4.55,48.25 -4.58,48.29 -4.53,48.31 -4.5,48.3 -4.4,48.29 -4.24,48.3 -4.3,48.35 -4.37,48.36 -4.39,48.37 -4.52,48.37 -4.58,48.36 -4.72,48.36 -4.75,48.41 -4.76,48.45 -4.72,48.54 -4.53,48.62 -4.06,48.71 -3.86,48.7 -3.71,48.71 -3.54,48.76 -3.47,48.81 -3.23,48.84 -3,48.79 -2.79,48.6 -2.69,48.54 -2.45,48.65 -2.08,48.65 -2,48.58 -1.97,48.63 -1.91,48.7 -1.85,48.67 -1.82,48.63 -1.44,48.64 -1.38,48.65 -1.48,48.7 -1.56,48.8 -1.58,49.2 -1.69,49.31 -1.81,49.49 -1.87,49.59 -1.87,49.63 -1.86,49.68 -1.7,49.68 -1.59,49.67 -1.37,49.71 -1.26,49.68 -1.27,49.6 -1.23,49.49 -1.19,49.45 -1.14,49.39 -0.96,49.39 -0.77,49.36 -0.52,49.36 -0.16,49.3 -0.01,49.33 0.14,49.4 0.42,49.45 0.44,49.47 0.28,49.46 0.13,49.51 0.11,49.56 0.13,49.6 0.19,49.7 0.62,49.86 0.92,49.91 1.24,50 1.41,50.09 1.51,50.21 1.55,50.23 1.59,50.25 1.55,50.29 1.58,50.74 1.61,50.82 1.67,50.88 1.77,50.93 1.91,50.99 2.45,51.07 2.53,51.1 2.54,51.05 2.58,50.99 2.6,50.96 2.58,50.91 2.6,50.88 2.67,50.81 2.76,50.75 2.84,50.71 2.86,50.72 2.92,50.73 3.02,50.77 3.11,50.78 3.16,50.75 3.18,50.73 3.23,50.66 3.25,50.59 3.27,50.53 3.32,50.51 3.48,50.5 3.59,50.48 3.63,50.46 3.67,50.33 3.69,50.31 3.72,50.32 3.75,50.34 3.79,50.35 3.86,50.34 3.95,50.34 4.04,50.32 4.17,50.25 4.17,50.22 4.15,50.18 4.13,50.14 4.16,50.13 4.19,50.09 4.19,50.05 4.15,50.02 4.14,50 4.14,49.98 4.15,49.97 4.18,49.96 4.37,49.95 4.55,49.96 4.66,50 4.67,50.05 4.71,50.1 4.77,50.14 4.82,50.15 4.86,50.14 4.79,49.96 4.84,49.91 4.85,49.85 4.87,49.79 4.93,49.79 5.01,49.78 5.06,49.76 5.12,49.72 5.21,49.69 5.28,49.68 5.3,49.65 5.36,49.62 5.43,49.55 5.51,49.51 5.54,49.51 5.61,49.53 5.71,49.54 5.79,49.54 5.82,49.5 5.9,49.49 5.93,49.48 5.96,49.45 6.01,49.45 6.08,49.45 6.12,49.49 6.18,49.5 6.24,49.49 6.28,49.48 6.35,49.45 6.38,49.46 6.46,49.44 6.54,49.4 6.56,49.35 6.58,49.32 6.61,49.29 6.73,49.16 6.78,49.15 6.82,49.17 6.85,49.2 6.89,49.21 6.96,49.2 7,49.18 7.02,49.12 7.04,49.11 7.07,49.12 7.12,49.13 7.2,49.11 7.31,49.13 7.4,49.15 7.45,49.15 7.53,49.09 7.61,49.06 7.8,49.04 8,49.01 8.08,48.99 8.13,48.97 8.14,48.89 8.12,48.87 7.92,48.7 7.84,48.64 7.8,48.55 7.76,48.41 7.71,48.28 7.62,48.16 7.58,48.06 7.61,48 7.59,47.91 7.54,47.77 7.53,47.67 7.57,47.61 7.62,47.59"]]},{"id":56,"c":[["4.22,51.39 4.3,51.36 4.37,51.36 4.4,51.37 4.38,51.43 4.44,51.46 4.5,51.47 4.53,51.45 4.59,51.42 4.64,51.42 4.76,51.49 4.78,51.48 4.81,51.45 4.82,51.43 4.82,51.41 4.85,51.4 4.94,51.41 4.99,51.45 5.03,51.47 5.06,51.45 5.07,51.41 5.1,51.35 5.21,51.28 5.31,51.26 5.43,51.27 5.48,51.29 5.51,51.28 5.54,51.24 5.61,51.2 5.75,51.17 5.8,51.15 5.83,51.13 5.82,51.09 5.75,50.99 5.74,50.96 5.75,50.95 5.74,50.93 5.65,50.87 5.64,50.84 5.67,50.81 5.69,50.78 5.75,50.76 5.8,50.75 5.89,50.75 5.99,50.75 6.01,50.73 6.12,50.68 6.15,50.64 6.24,50.6 6.17,50.55 6.18,50.52 6.2,50.5 6.29,50.49 6.34,50.45 6.35,50.4 6.36,50.32 6.18,50.23 6.12,50.14 6.12,50.12 6.11,50.12 6.09,50.15 6.05,50.15 5.98,50.17 5.87,50.08 5.82,50.01 5.79,49.96 5.74,49.92 5.74,49.88 5.74,49.86 5.73,49.83 5.73,49.81 5.79,49.76 5.81,49.73 5.88,49.65 5.86,49.61 5.84,49.58 5.82,49.55 5.79,49.54 5.71,49.54 5.61,49.53 5.54,49.51 5.51,49.51 5.43,49.55 5.36,49.62 5.3,49.65 5.28,49.68 5.21,49.69 5.12,49.72 5.06,49.76 5.01,49.78 4.93,49.79 4.87,49.79 4.85,49.85 4.84,49.91 4.79,49.96 4.86,50.14 4.82,50.15 4.77,50.14 4.71,50.1 4.67,50.05 4.66,50 4.55,49.96 4.37,49.95 4.18,49.96 4.15,49.97 4.14,49.98 4.14,50 4.15,50.02 4.19,50.05 4.19,50.09 4.16,50.13 4.13,50.14 4.15,50.18 4.17,50.22 4.17,50.25 4.04,50.32 3.95,50.34 3.86,50.34 3.79,50.35 3.75,50.34 3.72,50.32 3.69,50.31 3.67,50.33 3.63,50.46 3.59,50.48 3.48,50.5 3.32,50.51 3.27,50.53 3.25,50.59 3.23,50.66 3.18,50.73 3.16,50.75 3.11,50.78 3.02,50.77 2.92,50.73 2.86,50.72 2.84,50.71 2.76,50.75 2.67,50.81 2.6,50.88 2.58,50.91 2.6,50.96 2.58,50.99 2.54,51.05 2.53,51.1 2.96,51.26 3.22,51.35 3.35,51.38 3.38,51.29 3.4,51.26 3.43,51.25 3.47,51.24 3.52,51.26 3.58,51.29 3.68,51.28 3.76,51.25 3.78,51.23 3.83,51.21 3.9,51.21 4.04,51.25 4.17,51.31 4.21,51.35 4.22,51.39"]]}]

## 2) Explore the files
For each file we check the size, the first rows, the data types, missing values and the number of `-1` entries. In the government data, `-1` means "data missing", so it must be handled before any analysis.

In [ ]:
import pandas as pd
from IPython.display import display
raw = {k: pd.read_csv(f'dft-road-casualty-statistics-{k}-2025.csv', low_memory=False)
       for k in ['collision', 'vehicle', 'casualty']}

for name, df in raw.items():
    print('=' * 70)
    print(name.upper(), '| rows, columns =', df.shape, '| duplicate rows =', df.duplicated().sum())
    display(df.head(3))
    info = pd.DataFrame({'dtype': df.dtypes.astype(str), 'nulls': df.isna().sum(),
                         'minus_one (-1)': (df == -1).sum(), 'unique values': df.nunique()})
    display(info[(info['nulls'] > 0) | (info['minus_one (-1)'] > 0)])

In [ ]:
c0 = raw['collision']
print('Date range  :', c0['date'].min(), '->', c0['date'].max())
print('Lat range   :', c0['latitude'].min(), c0['latitude'].max())
print('Lon range   :', c0['longitude'].min(), c0['longitude'].max())
print('Collisions linked to vehicles :', c0['collision_index'].isin(raw['vehicle']['collision_index']).mean().round(3))
print('Collisions linked to casualties:', c0['collision_index'].isin(raw['casualty']['collision_index']).mean().round(3))
for col in ['collision_severity', 'road_type', 'junction_detail', 'speed_limit', 'light_conditions', 'weather_conditions']:
    print('\n' + col); print(c0[col].value_counts().head(10).to_string())

**Note:** the 2025 data changed the junction codes (13, 16 and 17 replace 3, 6 and 7). The cleaning step below handles this.

In [ ]:
import pandas as pd, numpy as np, base64, json, os
HERE = '.'

DIR = HERE
TEMPLATE = HERE + '/template.html'
LAND = HERE + '/land.json'
OUT = HERE + '/uk_road_safety_dashboard.html'
U = DIR + '/dft-road-casualty-statistics-%s-2025.csv'

## 3) Load the required columns

In [ ]:
# ---------- 1) Load ----------
c = pd.read_csv(U % 'collision', low_memory=False)
v = pd.read_csv(U % 'vehicle', usecols=['collision_index', 'vehicle_type'])
k = pd.read_csv(U % 'casualty', usecols=['collision_index', 'casualty_class'])
print('rows loaded  collisions/vehicles/casualties:', len(c), len(v), len(k))

## 4) Clean: unknown codes become "Unknown", dates and times are validated

In [ ]:
# ---------- 2) Clean ----------
n0 = len(c)
c = c.drop_duplicates('collision_index').reset_index(drop=True)
print('duplicate collisions removed:', n0 - len(c))

# code -> label (from the DfT data guide). Anything else (-1, 99, 9...) becomes "Unknown"
D = {
 'sev': ('collision_severity', {1: 'Fatal', 2: 'Serious', 3: 'Slight'}),
 'rt': ('road_type', {1: 'Roundabout', 2: 'One way street', 3: 'Dual carriageway', 6: 'Single carriageway', 7: 'Slip road', 12: 'One way / slip road'}),
 'jn': ('junction_detail', {0: 'No junction', 13: 'T / staggered', 16: 'Crossroads', 17: '4+ arms', 18: 'Private drive', 19: 'Other junction'}),
 'sp': ('speed_limit', {20: '20 mph', 30: '30 mph', 40: '40 mph', 50: '50 mph', 60: '60 mph', 70: '70 mph'}),
 'li': ('light_conditions', {1: 'Daylight', 4: 'Dark, lit', 5: 'Dark, unlit', 6: 'Dark, no lighting', 7: 'Dark, unknown'}),
 'we': ('weather_conditions', {1: 'Fine', 2: 'Rain', 3: 'Snow', 4: 'Fine + wind', 5: 'Rain + wind', 6: 'Snow + wind', 7: 'Fog / mist', 8: 'Other'}),
 'su': ('road_surface_conditions', {1: 'Dry', 2: 'Wet / damp', 3: 'Snow', 4: 'Frost / ice', 5: 'Flood', 6: 'Oil / diesel', 7: 'Mud'}),
 'ur': ('urban_or_rural_area', {1: 'Urban', 2: 'Rural'}),
}
cols, names = {}, {}
for key, (col, mp) in D.items():
    lab = list(mp.values()) + ['Unknown']
    idx = {code: i for i, code in enumerate(mp)}
    unk = (~c[col].isin(mp.keys())).sum()
    print(f'{col:26s} unknown/-1 -> "Unknown": {unk}')
    cols[key] = c[col].map(idx).fillna(len(lab) - 1).astype(int)
    names[key] = lab

# dates / times
dt = pd.to_datetime(c['date'], format='%d/%m/%Y', errors='coerce')
tm = pd.to_datetime(c['time'], format='%H:%M', errors='coerce')
print('bad dates:', dt.isna().sum(), '| bad times:', tm.isna().sum())
cols['dow'] = dt.dt.dayofweek.fillna(0).astype(int)
cols['mo'] = (dt.dt.month.fillna(1) - 1).astype(int)
cols['hr'] = tm.dt.hour.fillna(0).astype(int)

## 5) Merge vehicles and casualties into one row per collision

In [ ]:
# ---------- 3) Merge vehicles + casualties into one row per collision ----------
ped = set(k.loc[k.casualty_class == 3, 'collision_index'])
grp = lambda ts: set(v.loc[v.vehicle_type.isin(ts), 'collision_index'])
cyc, mot = grp([1]), grp([2, 3, 4, 5, 23, 97, 103, 104, 105, 106])
gds, bus = grp([19, 20, 21, 98, 113]), grp([10, 11, 108, 109])
ci = c['collision_index']
cols['inv'] = np.select([ci.isin(ped), ci.isin(cyc), ci.isin(mot), ci.isin(gds), ci.isin(bus)], [0, 1, 2, 3, 4], 5)
names['inv'] = ['Pedestrian', 'Pedal cycle', 'Motorcycle', 'Van / goods', 'Bus / coach', 'Cars & other']
cols['cas'] = c['number_of_casualties'].clip(0, 255)
cols['veh'] = c['number_of_vehicles'].clip(0, 255)

## 6) Validate the coordinates

In [ ]:
# ---------- 4) Coordinates ----------
ok = c.latitude.between(49.8, 61) & c.longitude.between(-8.7, 2)
print('missing / out-of-range coordinates (kept in charts, hidden on map):', (~ok).sum())
la = np.where(ok, ((c.latitude - 49.5) / 11.5 * 65535).round(), 0).astype('<u2')
lo = np.where(ok, ((c.longitude + 9) / 11 * 65535).round(), 0).astype('<u2')

## 7) Quick analysis

In [ ]:
# ---------- 5) Quick analysis printout ----------
print('\nFatal+serious share by junction type:')
t = pd.DataFrame({'jn': [names['jn'][i] for i in cols['jn']], 'ks': np.array(cols['sev']) < 2})
print((t.groupby('jn').ks.agg(['mean', 'size']).round(3)).sort_values('mean', ascending=False).to_string())

## 8) Build the dashboard

In [ ]:
# ---------- 6) Encode + build ----------
ORDER = ['sev', 'dow', 'hr', 'mo', 'rt', 'jn', 'sp', 'li', 'we', 'su', 'ur', 'inv', 'cas', 'veh']
buf = b''.join(np.asarray(cols[x]).astype('u1').tobytes() for x in ORDER) + la.tobytes() + lo.tobytes()
meta = {x: names[x] for x in names}
html = open(TEMPLATE, encoding='utf-8').read()
html = html.replace('__DATA__', base64.b64encode(buf).decode()).replace('__N__', str(len(c))).replace('__META__', json.dumps(meta)).replace('__LAND__', open(LAND).read())
open(OUT, 'w', encoding='utf-8').write(html)
print(f'\nBuilt {OUT}  ({len(html)/1e6:.2f} MB, {len(c):,} collisions)')

## 9) Download the dashboard
The file opens in any browser, works offline, and can be shared as a single attachment.

In [ ]:
files.download('uk_road_safety_dashboard.html')